In [1]:
import pandas as pd
import numpy as np

In [2]:
data=pd.read_csv('startup_funding.csv')

In [3]:
df=pd.DataFrame(data)

In [4]:
df.head(
)


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [5]:
# --- DATA CLEANING ---

In [6]:
df.dtypes


,0
Sr No,int64
Date dd/mm/yyyy,object
Startup Name,object
Industry Vertical,object
SubVertical,object
City Location,object
Investors Name,object
InvestmentnType,object
Amount in USD,object
Remarks,object


In [7]:
def clean_amount(val):
  # Skip if already null
  if pd.isna(val):
    return np.nan
  val = str(val).lower().strip()
  # Strip commas and currency symbols
  val = val.replace(',', '').replace('$', '').strip()
  # Treat undisclosed/unknown/empty as null
  if 'undisclosed' in val or 'unknown' in val or val == '':
    return np.nan
  try:
    return float(val)
  except ValueError:
    return np.nan


In [8]:
def clean_startup_name(name):
  if pd.isna(name):
    return "UNKNOWN"
  name = str(name).strip()
  # Strip URL prefixes — some rows had full URLs instead of names
  if "http" in name or "www." in name:
    name = name.replace("https://", "").replace("http://", "").replace("www.", "")
    name = name.split('.')[0].capitalize().strip()
  return name


In [9]:
# Apply name cleaning to Startup Name column
df['Startup Name'] = df['Startup Name'].apply(clean_startup_name)


In [10]:
arr=df['Investors Name'].unique()

In [11]:
print(df.columns.tolist())

['Sr No', 'Date dd/mm/yyyy', 'Startup Name', 'Industry Vertical', 'SubVertical', 'City  Location', 'Investors Name', 'InvestmentnType', 'Amount in USD', 'Remarks']


In [12]:
# Normalize inconsistent city spellings to a single standard
city_mapping = {
    'Bengaluru': 'Bangalore',
    'Gurgaon': 'Gurugram',
    'New Delhi': 'Delhi'
}
df['City  Location'] = df['City  Location'].str.strip().replace(city_mapping)

# Fill missing cities with Unknown
df['City  Location'] = df['City  Location'].fillna('Unknown')


In [13]:
# Fix date format inconsistencies — replace dots and double slashes with single slash
df['Date dd/mm/yyyy'] = df['Date dd/mm/yyyy'].str.replace('.', '/', regex=False)
df['Date dd/mm/yyyy'] = df['Date dd/mm/yyyy'].str.replace('//', '/', regex=False)
df['Date dd/mm/yyyy'] = df['Date dd/mm/yyyy'].str.strip()

# Parse to datetime; unparseable values become NaT
df['Clean_Date'] = pd.to_datetime(df['Date dd/mm/yyyy'], format='%d/%m/%Y', errors='coerce')

print(df[['Clean_Date', 'Startup Name', 'City  Location', 'Amount in USD']].head())


  Clean_Date  Startup Name City  Location Amount in USD
0 2020-01-09        BYJU’S      Bangalore  20,00,00,000
1 2020-01-13        Shuttl       Gurugram     80,48,394
2 2020-01-09     Mamaearth      Bangalore   1,83,58,860
3 2020-01-02  Wealthbucket          Delhi     30,00,000
4 2020-01-02        Fashor         Mumbai     18,00,000


In [14]:
df.columns


Index(['Sr No', 'Date dd/mm/yyyy', 'Startup Name', 'Industry Vertical',
       'SubVertical', 'City  Location', 'Investors Name', 'InvestmentnType',
       'Amount in USD', 'Remarks', 'Clean_Date'],
      dtype='object')

In [15]:
# Drop the original messy date column — we have Clean_Date now
df.drop('Date dd/mm/yyyy', axis=1, inplace=True)


In [16]:
# Check how many funding rounds each startup appears in
print(df['Startup Name'].value_counts().head(20))


Startup Name
Swiggy                  8
Ola Cabs                8
Paytm                   7
Medinfi                 6
NoBroker                6
Meesho                  6
UrbanClap               6
Nykaa                   6
Capital Float           5
Uniphore                5
Jugnoo                  5
Flipkart                5
Grofers                 5
Moglix                  5
Toppr                   5
Zomato                  4
BigBasket               4
Ola                     4
Byju\\xe2\\x80\\x99s    4
Udaan                   4
Name: count, dtype: int64


In [17]:
import sqlite3

In [18]:
# Build the startups dimension table — one row per unique startup
startups_df = df[['Startup Name', 'City  Location', 'Industry Vertical']].copy()
startups_df = startups_df.drop_duplicates(subset=['Startup Name'])
startups_df = startups_df.reset_index(drop=True)
# Assign a surrogate key starting from 1
startups_df['startup_id'] = startups_df.index + 1

print('Unique startups:', len(startups_df))
print(startups_df.head())


Unique startups: 2459
   Startup Name City  Location    Industry Vertical  startup_id
0        BYJU’S      Bangalore               E-Tech           1
1        Shuttl       Gurugram       Transportation           2
2     Mamaearth      Bangalore           E-commerce           3
3  Wealthbucket          Delhi              FinTech           4
4        Fashor         Mumbai  Fashion and Apparel           5


In [19]:
# Merge startup_id into the main dataframe
merged_df = df.merge(startups_df[['Startup Name', 'startup_id']], on='Startup Name', how='left')

# Build the funding fact table with only relevant columns
funding_df = merged_df[['startup_id', 'Clean_Date', 'Investors Name', 'InvestmentnType', 'Amount in USD']].copy()

# Rename to SQL-friendly lowercase column names
funding_df.columns = ['startup_id', 'funding_date', 'investor_name', 'investment_type', 'amount_usd']
startups_df.columns = ['startup_name', 'city_location', 'industry', 'startup_id']

print(funding_df.head())


   startup_id funding_date              investor_name       investment_type  \
0           1   2020-01-09    Tiger Global Management  Private Equity Round   
1           2   2020-01-13  Susquehanna Growth Equity              Series C   
2           3   2020-01-09      Sequoia Capital India              Series B   
3           4   2020-01-02             Vinod Khatumal          Pre-series A   
4           5   2020-01-02    Sprout Venture Partners            Seed Round   

     amount_usd  
0  20,00,00,000  
1     80,48,394  
2   1,83,58,860  
3     30,00,000  
4     18,00,000  


In [20]:
merged_df

,Sr No,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks,Clean_Date,startup_id
0,1,BYJU’S,E-Tech,E-learning,Bangalore,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN,2020-01-09,1
1,2,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,"80,48,394",NaN,2020-01-13,2
2,3,Mamaearth,E-commerce,Retailer of baby and toddler products,Bangalore,Sequoia Capital India,Series B,"1,83,58,860",NaN,2020-01-09,3
3,4,Wealthbucket,FinTech,Online Investment,Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN,2020-01-02,4
4,5,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN,2020-01-02,5
...,...,...,...,...,...,...,...,...,...,...,...
3039,3040,Printvenue,NaN,NaN,Unknown,Asia Pacific Internet Group,Private Equity,"45,00,000",NaN,2015-01-29,2456
3040,3041,Graphene,NaN,NaN,Unknown,KARSEMVEN Fund,Private Equity,"8,25,000",Govt backed VC Fund,2015-01-29,2457
3041,3042,Mad Street Den,NaN,NaN,Unknown,"Exfinity Fund, GrowX Ventures.",Private Equity,"15,00,000",NaN,2015-01-30,149
3042,3043,Simplotel,NaN,NaN,Unknown,MakeMyTrip,Private Equity,NaN,"Strategic Funding, Minority stake",2015-01-30,2458


In [21]:
merged_df['startup_id'].value_counts()

,count
startup_id,
333,8
63,8
31,7
614,6
72,6
...,...
2442,1
2443,1
10,1


In [22]:
startups_df.head(5)

,startup_name,city_location,industry,startup_id
0,BYJU’S,Bangalore,E-Tech,1
1,Shuttl,Gurugram,Transportation,2
2,Mamaearth,Bangalore,E-commerce,3
3,Wealthbucket,Delhi,FinTech,4
4,Fashor,Mumbai,Fashion and Apparel,5


In [23]:
funding_df['startup_id']

,startup_id
0,1
1,2
2,3
3,4
4,5
...,...
3039,2456
3040,2457
3041,149
3042,2458


In [24]:
# Remove commas from amount strings, then cast to float
# Non-numeric values like 'undisclosed' become NaN
funding_df['amount_usd'] = funding_df['amount_usd'].astype(str).str.replace(',', '')
funding_df['amount_usd'] = pd.to_numeric(funding_df['amount_usd'], errors='coerce')


In [25]:
df['City  Location'].unique()

array(['Bangalore', 'Gurugram', 'Delhi', 'Mumbai', 'Chennai', 'Pune',
       'Noida', 'Faridabad', 'San Francisco', 'San Jose,', 'Amritsar',
       'Kormangala', 'Tulangan', 'Hyderabad', 'Burnsville', 'Menlo Park',
       'Palo Alto', 'Santa Monica', 'Singapore', 'Taramani', 'Andheri',
       'Chembur', 'Nairobi', 'Haryana', 'New York', 'Karnataka',
       'Mumbai/Bengaluru', 'Bhopal', 'Bengaluru and Gurugram',
       'India/Singapore', 'Jaipur', 'India/US', 'Nagpur', 'Indore',
       'New York, Bengaluru', 'California', 'India', 'Ahemadabad',
       'Rourkela', 'Srinagar', 'Bhubneswar', 'Chandigarh',
       'Delhi & Cambridge', 'Kolkatta', 'Kolkata', 'Coimbatore',
       'Udaipur', 'Unknown', 'Ahemdabad', 'Bhubaneswar', 'Ahmedabad',
       'Surat', 'Goa', 'Uttar Pradesh', 'Nw Delhi', 'Gaya', 'Vadodara',
       'Trivandrum', 'Missourie', 'Panaji', 'Gwalior', 'Karur', 'Udupi',
       'Kochi', 'Agra', 'Bangalore/ Bangkok', 'Hubli', 'Kerala',
       'Kozhikode', 'US', 'Siliguri', 'USA', '

In [26]:
import sqlite3

In [27]:
conn = sqlite3.connect('startup_db.sqlite')

In [28]:
# Write startups table to SQLite
startups_df.to_sql('startups', conn, if_exists='replace', index=False)


2459

In [29]:
print("Tables loaded into startup_db.sqlite")


Tables loaded into startup_db.sqlite


In [30]:
# Write funding rounds table to SQLite
funding_df.to_sql('funding_rounds', conn, if_exists='replace', index=False)


3044

In [31]:
# Total funding raised per city — top 10
query = """
SELECT
    s.city_location,
    SUM(f.amount_usd) AS total_funding
FROM startups s
JOIN funding_rounds f ON s.startup_id = f.startup_id
GROUP BY s.city_location
ORDER BY total_funding DESC
LIMIT 10;
"""
city_funding_df = pd.read_sql(query, conn)
print(city_funding_df)


  city_location  total_funding
0     Bangalore   1.639725e+10
1        Mumbai   5.004360e+09
2      Gurugram   4.063884e+09
3         Noida   3.386964e+09
4         Delhi   2.585132e+09
5    Kormangala   9.527000e+08
6       Unknown   8.096179e+08
7       Chennai   7.147170e+08
8          Pune   6.927420e+08
9    Menlo Park   4.500000e+08


In [32]:
# All deals made by Tiger Global Management, sorted by size
query = """
SELECT
    s.startup_name,
    f.investor_name,
    f.amount_usd
FROM startups s
JOIN funding_rounds f ON s.startup_id = f.startup_id
WHERE f.investor_name = 'Tiger Global Management'
ORDER BY f.amount_usd DESC;
"""
tiger_df = pd.read_sql(query, conn)
print(tiger_df.head())


  startup_name            investor_name   amount_usd
0       BYJU’S  Tiger Global Management  200000000.0
1       Zenoti  Tiger Global Management   50000000.0
2  Grey Orange  Tiger Global Management   30000000.0
3     OkCredit  Tiger Global Management   15500000.0
4    INDwealth  Tiger Global Management   15000000.0


In [33]:
# Number of funding rounds per city — top 10
query = """
SELECT
    s.city_location,
    COUNT(*) AS number_of_funding_rounds
FROM startups s
JOIN funding_rounds f ON s.startup_id = f.startup_id
GROUP BY s.city_location
ORDER BY number_of_funding_rounds DESC
LIMIT 10;
"""
city_count_df = pd.read_sql(query, conn)
print(city_count_df)


  city_location  number_of_funding_rounds
0     Bangalore                       850
1        Mumbai                       572
2         Delhi                       445
3      Gurugram                       357
4       Unknown                       130
5          Pune                       112
6     Hyderabad                       105
7         Noida                        96
8       Chennai                        96
9     Ahmedabad                        34


In [34]:
# Round count and largest single deal per city — top 2
query = """
SELECT
    s.city_location,
    COUNT(*) AS total_events,
    MAX(f.amount_usd) AS biggest_single_investment
FROM startups s
JOIN funding_rounds f ON s.startup_id = f.startup_id
GROUP BY s.city_location
ORDER BY total_events DESC
LIMIT 2;
"""
city_insights_df = pd.read_sql(query, conn)
pd.options.display.float_format = '{:,.0f}'.format
print(city_insights_df)


  city_location  total_events  biggest_single_investment
0     Bangalore           850              3,900,000,000
1        Mumbai           572                600,000,000


In [35]:
# CTE: find the startup that received the largest deal in each city
query = """
WITH CityMax AS (
    SELECT
        s.city_location,
        MAX(f.amount_usd) AS biggest_check
    FROM startups s
    JOIN funding_rounds f ON s.startup_id = f.startup_id
    GROUP BY s.city_location
)
SELECT
    cm.city_location,
    s.startup_name,
    cm.biggest_check
FROM CityMax cm
JOIN startups s ON cm.city_location = s.city_location
JOIN funding_rounds f ON s.startup_id = f.startup_id AND f.amount_usd = cm.biggest_check
ORDER BY cm.biggest_check DESC
LIMIT 5;
"""
top_startups_df = pd.read_sql(query, conn)
pd.options.display.float_format = '{:,.0f}'.format
print(top_startups_df)


  city_location      startup_name  biggest_check
0     Bangalore  Rapido Bike Taxi  3,900,000,000
1         Noida             Paytm  1,400,000,000
2        Mumbai        True North    600,000,000
3         Delhi          Snapdeal    500,000,000
4    Menlo Park             GOQii    450,000,000


In [36]:
# Rank every startup within its city by funding amount
# PARTITION BY resets the rank counter for each city
query = """
SELECT
    s.city_location,
    s.startup_name,
    f.amount_usd,
    RANK() OVER(PARTITION BY s.city_location ORDER BY f.amount_usd DESC) AS city_rank
FROM startups s
JOIN funding_rounds f ON s.startup_id = f.startup_id
"""
ranked_startups_df = pd.read_sql(query, conn)

# Spot check — top 3 in Bangalore
bangalore_top_3 = ranked_startups_df[
    (ranked_startups_df['city_location'] == 'Bangalore') &
    (ranked_startups_df['city_rank'] <= 3)
]
print(bangalore_top_3)


   city_location      startup_name    amount_usd  city_rank
45     Bangalore  Rapido Bike Taxi 3,900,000,000          1
46     Bangalore          Flipkart 2,500,000,000          2
47     Bangalore          Flipkart 1,400,000,000          3


In [37]:
import pandas as pd

# Pull final tables from the database and export as CSVs for Power BI
clean_startups = pd.read_sql("SELECT * FROM startups", conn)
clean_funding = pd.read_sql("SELECT * FROM funding_rounds", conn)

clean_startups.to_csv("dim_startups.csv", index=False)
clean_funding.to_csv("fact_funding.csv", index=False)

print("Exported dim_startups.csv and fact_funding.csv")


Exported dim_startups.csv and fact_funding.csv
